In [1]:
import pandas as pd
import numpy as np
import os
from sqlalchemy import create_engine
from urllib.parse import quote_plus

RAW       = r"D:\Projects\End-to-end projects\11. Fashion Trend Inventory\Data\Raw"
PROCESSED = r"D:\Projects\End-to-end projects\11. Fashion Trend Inventory\Data\Processed"

print("Setup complete ✅")

Setup complete ✅


In [2]:
# ── Load Raw ───────────────────────────────────────────────────
df_products  = pd.read_csv(os.path.join(RAW, 'products.csv'))
df_trends    = pd.read_csv(os.path.join(RAW, 'trend_signals.csv'))
df_inventory = pd.read_csv(os.path.join(RAW, 'inventory.csv'))
df_sales     = pd.read_csv(os.path.join(RAW, 'sales_transactions.csv'))
df_po        = pd.read_csv(os.path.join(RAW, 'purchase_orders.csv'))
df_social    = pd.read_csv(os.path.join(RAW, 'social_signals.csv'))
df_mapping   = pd.read_csv(os.path.join(RAW, 'trend_sku_mapping.csv'))

# ── Load Processed ─────────────────────────────────────────────
df_ml        = pd.read_csv(os.path.join(PROCESSED, 'ml_master_scores.csv'))
df_gap       = pd.read_csv(os.path.join(PROCESSED, 'gap_analysis.csv'))

# ── Parse Dates ────────────────────────────────────────────────
df_sales['sale_date']         = pd.to_datetime(df_sales['sale_date'])
df_inventory['snapshot_date'] = pd.to_datetime(df_inventory['snapshot_date'])
df_trends['week_start_date']  = pd.to_datetime(df_trends['week_start_date'])
df_po['po_date']              = pd.to_datetime(df_po['po_date'])
df_po['actual_delivery']      = pd.to_datetime(df_po['actual_delivery'])
df_social['week_start_date']  = pd.to_datetime(df_social['week_start_date'])

print("All data loaded ✅")
print(f"\n  Raw tables    : 7")
print(f"  Processed     : 2")
print(f"  Total rows    : {sum([len(df_products),len(df_trends),len(df_inventory),len(df_sales),len(df_po),len(df_social),len(df_mapping)]):,}")

All data loaded ✅

  Raw tables    : 7
  Processed     : 2
  Total rows    : 292,740


In [3]:
# ── Executive KPI Table ────────────────────────────────────────
# Single row summary for KPI cards in Power BI

kpi = {
    'total_revenue_cr'      : round(df_sales['revenue'].sum() / 1e7, 2),
    'total_gross_profit_cr' : round(df_sales['gross_profit'].sum() / 1e7, 2),
    'avg_gross_margin_pct'  : round(df_sales['gross_margin_pct'].mean(), 2),
    'total_orders'          : len(df_sales),
    'total_units_sold'      : int(df_sales['quantity'].sum()),
    'avg_order_value'       : round(df_sales['revenue'].mean(), 2),
    'avg_discount_pct'      : round(df_sales['discount_pct'].mean(), 2),
    'overall_return_rate'   : round(df_sales['is_returned'].mean() * 100, 2),
    'total_returns'         : int(df_sales['is_returned'].sum()),
    'cities_covered'        : df_sales['city'].nunique(),
    'channels_active'       : df_sales['channel'].nunique(),
    'skus_with_sales'       : df_sales['sku_id'].nunique(),
    'total_skus'            : len(df_products),
    'dead_stock_skus'       : int((df_ml['sku_status'] == 'DEAD STOCK').sum()),
    'dead_stock_pct'        : round(
                                (df_ml['sku_status']=='DEAD STOCK').mean()*100, 2),
    'stockout_weeks_total'  : int(df_inventory['stockout_flag'].sum()),
    'avg_weeks_cover'       : round(df_inventory['weeks_of_cover'].mean(), 2),
    'total_po_value_cr'     : round(df_po['po_value'].sum() / 1e7, 2),
    'avg_lead_time_days'    : round(df_po['lead_time_days'].mean(), 1),
    'supplier_delay_rate'   : round(df_po['is_delayed'].mean() * 100, 2),
    'active_trends'         : df_trends['trend_name'].nunique(),
    'total_social_mentions' : int(df_social['mentions'].sum()),
}

df_kpi = pd.DataFrame([kpi])
df_kpi.to_csv(os.path.join(PROCESSED, 'pbi_01_kpi_summary.csv'), index=False)

print("KPI Summary exported ✅")
print(f"\n  Key Metrics:")
print(f"    Total Revenue        : ₹{kpi['total_revenue_cr']} Cr")
print(f"    Gross Profit         : ₹{kpi['total_gross_profit_cr']} Cr")
print(f"    Avg Gross Margin     : {kpi['avg_gross_margin_pct']}%")
print(f"    Return Rate          : {kpi['overall_return_rate']}%")
print(f"    Dead Stock SKUs      : {kpi['dead_stock_skus']} ({kpi['dead_stock_pct']}%)")
print(f"    Avg Lead Time        : {kpi['avg_lead_time_days']} days")
print(f"    Supplier Delay Rate  : {kpi['supplier_delay_rate']}%")

KPI Summary exported ✅

  Key Metrics:
    Total Revenue        : ₹30.4 Cr
    Gross Profit         : ₹15.46 Cr
    Avg Gross Margin     : 49.68%
    Return Rate          : 19.43%
    Dead Stock SKUs      : 348 (29.74%)
    Avg Lead Time        : 60.2 days
    Supplier Delay Rate  : 59.52%


In [4]:
# ── Weekly Revenue Time Series ─────────────────────────────────
# Used for: Revenue trend line, cumulative revenue, margin trend

weekly_revenue = (df_sales
    .groupby(pd.Grouper(key='sale_date', freq='W'))
    .agg(
        weekly_revenue    = ('revenue','sum'),
        weekly_profit     = ('gross_profit','sum'),
        weekly_orders     = ('sale_id','count'),
        weekly_units      = ('quantity','sum'),
        avg_margin        = ('gross_margin_pct','mean'),
        avg_discount      = ('discount_pct','mean'),
        weekly_returns    = ('is_returned','sum'),
    )
    .reset_index()
    .rename(columns={'sale_date':'week_start'})
)

# Additional calculated columns
weekly_revenue['return_rate']         = (
    weekly_revenue['weekly_returns'] /
    weekly_revenue['weekly_orders'] * 100
).round(2)

weekly_revenue['revenue_rolling4']    = (
    weekly_revenue['weekly_revenue'].rolling(4).mean().round(2)
)

weekly_revenue['cumulative_revenue']  = (
    weekly_revenue['weekly_revenue'].cumsum().round(2)
)

weekly_revenue['wow_growth_pct']      = (
    weekly_revenue['weekly_revenue'].pct_change() * 100
).round(2)

weekly_revenue['year']  = weekly_revenue['week_start'].dt.year
weekly_revenue['month'] = weekly_revenue['week_start'].dt.month
weekly_revenue['quarter'] = weekly_revenue['week_start'].dt.quarter

weekly_revenue = weekly_revenue.fillna(0)
weekly_revenue.to_csv(
    os.path.join(PROCESSED, 'pbi_02_weekly_revenue.csv'), index=False)

print(f"Weekly Revenue exported ✅")
print(f"  Rows : {len(weekly_revenue):,}")
print(f"  Date range: {weekly_revenue['week_start'].min().date()} "
      f"→ {weekly_revenue['week_start'].max().date()}")
print(f"  Total Revenue: ₹{weekly_revenue['weekly_revenue'].sum()/1e7:.2f} Cr")

Weekly Revenue exported ✅
  Rows : 106
  Date range: 2023-01-01 → 2025-01-05
  Total Revenue: ₹30.40 Cr


In [5]:
# ── Brand Performance Table ────────────────────────────────────
brand_perf = (df_sales.groupby(['brand','category'])
    .agg(
        total_revenue    = ('revenue','sum'),
        total_profit     = ('gross_profit','sum'),
        total_orders     = ('sale_id','count'),
        total_units      = ('quantity','sum'),
        avg_margin       = ('gross_margin_pct','mean'),
        avg_discount     = ('discount_pct','mean'),
        avg_mrp          = ('mrp','mean'),
        avg_selling_price= ('selling_price','mean'),
        total_returns    = ('is_returned','sum'),
        return_rate      = ('is_returned','mean'),
    )
    .round(2)
    .reset_index()
)

brand_perf['revenue_share_pct'] = (
    brand_perf['total_revenue'] /
    brand_perf['total_revenue'].sum() * 100
).round(2)

brand_perf['profit_margin_pct'] = (
    brand_perf['total_profit'] /
    brand_perf['total_revenue'] * 100
).round(2)

brand_perf['return_rate_pct'] = (
    brand_perf['return_rate'] * 100
).round(2)

# ── Channel Performance Table ──────────────────────────────────
channel_perf = (df_sales.groupby(['channel','category'])
    .agg(
        total_revenue    = ('revenue','sum'),
        total_profit     = ('gross_profit','sum'),
        total_orders     = ('sale_id','count'),
        total_units      = ('quantity','sum'),
        avg_margin       = ('gross_margin_pct','mean'),
        avg_discount     = ('discount_pct','mean'),
        total_returns    = ('is_returned','sum'),
        return_rate      = ('is_returned','mean'),
    )
    .round(2)
    .reset_index()
)

channel_perf['revenue_per_order'] = (
    channel_perf['total_revenue'] /
    channel_perf['total_orders']
).round(2)

channel_perf['return_rate_pct'] = (
    channel_perf['return_rate'] * 100
).round(2)

# ── Save ───────────────────────────────────────────────────────
brand_perf.to_csv(
    os.path.join(PROCESSED, 'pbi_03_brand_performance.csv'), index=False)
channel_perf.to_csv(
    os.path.join(PROCESSED, 'pbi_04_channel_performance.csv'), index=False)

print(f"Brand Performance exported   ✅  {len(brand_perf):,} rows")
print(f"Channel Performance exported ✅  {len(channel_perf):,} rows")
print(f"\n  Top 3 Brands by Revenue:")
top3 = (brand_perf.groupby('brand')['total_revenue']
        .sum().sort_values(ascending=False).head(3))
for brand, rev in top3.items():
    print(f"    {brand:<20} ₹{rev/1e6:.1f}M")

Brand Performance exported   ✅  60 rows
Channel Performance exported ✅  36 rows

  Top 3 Brands by Revenue:
    H&M India            ₹40.9M
    Urbanic              ₹38.7M
    Snitch               ₹35.9M


In [6]:
# ── Trend Phase Summary ────────────────────────────────────────
trend_phase_summary = (df_trends.groupby(['trend_name','trend_phase','decay_speed'])
    .agg(
        avg_trend_score   = ('trend_score','mean'),
        max_trend_score   = ('trend_score','max'),
        avg_search_vol    = ('search_volume_idx','mean'),
        avg_social_mention= ('social_mentions','mean'),
        weeks_in_phase    = ('week_number','count'),
        peak_week         = ('peak_week','first'),
        max_score         = ('max_score','first'),
    )
    .round(4)
    .reset_index()
)

# ── Weekly Trend Signals (for time series in Power BI) ────────
trend_weekly = (df_trends[['trend_name','week_start_date','week_number',
                            'year','month','trend_score','trend_phase',
                            'search_volume_idx','social_mentions',
                            'decay_speed','peak_week','max_score']]
                .copy())

trend_weekly['is_peak'] = (
    trend_weekly['week_number'] == trend_weekly['peak_week']
).astype(int)

trend_weekly['trend_momentum'] = (trend_weekly
    .sort_values(['trend_name','week_number'])
    .groupby('trend_name')['trend_score']
    .diff()
    .fillna(0)
    .round(4))

# ── Social Platform Summary ────────────────────────────────────
social_summary = (df_social.groupby(['platform','trend_name','city_tier'])
    .agg(
        total_mentions    = ('mentions','sum'),
        avg_mentions      = ('mentions','mean'),
        total_reach       = ('hashtag_reach','sum'),
        avg_engagement    = ('engagement_rate','mean'),
        peak_week_signals = ('is_peak_week','sum'),
    )
    .round(4)
    .reset_index()
)

# ── Save ───────────────────────────────────────────────────────
trend_phase_summary.to_csv(
    os.path.join(PROCESSED, 'pbi_05_trend_phase_summary.csv'), index=False)
trend_weekly.to_csv(
    os.path.join(PROCESSED, 'pbi_06_trend_weekly.csv'), index=False)
social_summary.to_csv(
    os.path.join(PROCESSED, 'pbi_07_social_summary.csv'), index=False)

print(f"Trend Phase Summary exported  ✅  {len(trend_phase_summary):,} rows")
print(f"Trend Weekly exported         ✅  {len(trend_weekly):,} rows")
print(f"Social Summary exported       ✅  {len(social_summary):,} rows")

print(f"\n  Trend Phase Distribution:")
print(df_trends['trend_phase'].value_counts().to_string())

Trend Phase Summary exported  ✅  93 rows
Trend Weekly exported         ✅  2,080 rows
Social Summary exported       ✅  360 rows

  Trend Phase Distribution:
trend_phase
Declining    456
Dead         422
Emerging     404
Peaking      402
Rising       396


In [7]:
# ── Inventory Health Summary ───────────────────────────────────
inv_health_summary = (df_inventory
    .groupby(['category','brand','inventory_health'])
    .agg(
        avg_closing_stock  = ('closing_stock','mean'),
        avg_weeks_cover    = ('weeks_of_cover','mean'),
        total_inv_value    = ('inventory_value','sum'),
        stockout_weeks     = ('stockout_flag','sum'),
        total_snapshots    = ('inventory_id','count'),
    )
    .round(2)
    .reset_index()
)

inv_health_summary['stockout_rate_pct'] = (
    inv_health_summary['stockout_weeks'] /
    inv_health_summary['total_snapshots'] * 100
).round(2)

# ── Weekly Inventory Trend ─────────────────────────────────────
inv_weekly = (df_inventory
    .groupby(['snapshot_date','category'])
    .agg(
        avg_closing_stock  = ('closing_stock','mean'),
        avg_weeks_cover    = ('weeks_of_cover','mean'),
        total_inv_value    = ('inventory_value','sum'),
        stockout_count     = ('stockout_flag','sum'),
        critical_count     = ('inventory_health',
                              lambda x: (x=='Critical').sum()),
        overstock_count    = ('inventory_health',
                              lambda x: (x=='Overstock').sum()),
        healthy_count      = ('inventory_health',
                              lambda x: (x=='Healthy').sum()),
    )
    .round(2)
    .reset_index()
)

# ── Gap Analysis Export ────────────────────────────────────────
gap_export = df_gap[[
    'sku_id','category','brand','segment',
    'mrp','gross_margin_pct','launch_season',
    'closing_stock','weeks_of_cover','inventory_health',
    'stockout_flag','inventory_value',
    'avg_trend_score','max_trend_score',
    'gap_score','action_flag'
]].copy()

gap_export['inv_value_at_risk'] = (
    gap_export['inventory_value'] *
    (gap_export['avg_trend_score'] < 0.3).astype(int)
).round(2)

# ── ML Scores Export ───────────────────────────────────────────
ml_export = df_ml[[
    'sku_id','category','brand','segment','mrp',
    'avg_weekly_qty','avg_trend_score','max_trend_score',
    'avg_closing_stock','avg_weeks_cover','total_stockouts',
    'avg_lead_time','delay_rate',
    'avg_discount','avg_margin','return_rate',
    'total_revenue','total_units_sold',
    'stockout_risk_score','deadstock_risk_score',
    'avg_buy_opportunity','sku_status'
]].copy()

# ── Save ───────────────────────────────────────────────────────
inv_health_summary.to_csv(
    os.path.join(PROCESSED, 'pbi_08_inventory_health.csv'), index=False)
inv_weekly.to_csv(
    os.path.join(PROCESSED, 'pbi_09_inventory_weekly.csv'), index=False)
gap_export.to_csv(
    os.path.join(PROCESSED, 'pbi_10_gap_analysis.csv'), index=False)
ml_export.to_csv(
    os.path.join(PROCESSED, 'pbi_11_ml_scores.csv'), index=False)

print(f"Inventory Health Summary exported  ✅  {len(inv_health_summary):,} rows")
print(f"Inventory Weekly exported          ✅  {len(inv_weekly):,} rows")
print(f"Gap Analysis exported              ✅  {len(gap_export):,} rows")
print(f"ML Scores exported                 ✅  {len(ml_export):,} rows")

print(f"\n  Inventory Health Distribution:")
print(df_inventory['inventory_health'].value_counts().to_string())
print(f"\n  SKU Status Distribution:")
print(df_ml['sku_status'].value_counts().to_string())

Inventory Health Summary exported  ✅  240 rows
Inventory Weekly exported          ✅  630 rows
Gap Analysis exported              ✅  757 rows
ML Scores exported                 ✅  1,170 rows

  Inventory Health Distribution:
inventory_health
Overstock    67376
Healthy      29895
Low          19706
Critical      5873

  SKU Status Distribution:
sku_status
STABLE        822
DEAD STOCK    348


In [8]:
# ── Supplier Performance ───────────────────────────────────────
supplier_perf = (df_po.groupby(['supplier','category'])
    .agg(
        total_pos          = ('po_id','count'),
        avg_lead_time      = ('lead_time_days','mean'),
        avg_delay_days     = ('delay_days','mean'),
        delay_rate         = ('is_delayed','mean'),
        avg_fulfillment    = ('fulfillment_rate','mean'),
        total_po_value     = ('po_value','sum'),
        delayed_pos        = ('is_delayed','sum'),
        max_delay          = ('delay_days','max'),
    )
    .round(2)
    .reset_index()
)

supplier_perf['delay_rate_pct'] = (
    supplier_perf['delay_rate'] * 100
).round(2)

supplier_perf['risk_score'] = (
    supplier_perf['delay_rate'] * 0.5 +
    (supplier_perf['avg_delay_days'] / 45) * 0.3 +
    (1 - supplier_perf['avg_fulfillment']/100) * 0.2
).round(4)

supplier_perf['supplier_grade'] = supplier_perf['risk_score'].apply(
    lambda x: 'A — Reliable'   if x < 0.25
         else 'B — Average'    if x < 0.40
         else 'C — Risky'      if x < 0.55
         else 'D — Critical'
)

# ── Monthly PO Timeline ────────────────────────────────────────
df_po['po_month']      = df_po['po_date'].dt.to_period('M').astype(str)
df_po['delivery_month']= df_po['actual_delivery'].dt.to_period('M').astype(str)

po_timeline = (df_po.groupby(['po_month','supplier','category'])
    .agg(
        total_pos       = ('po_id','count'),
        total_value     = ('po_value','sum'),
        avg_lead_time   = ('lead_time_days','mean'),
        delayed_count   = ('is_delayed','sum'),
        avg_delay       = ('delay_days','mean'),
    )
    .round(2)
    .reset_index()
)

# ── Geographic Performance ─────────────────────────────────────
geo_perf = (df_sales.groupby(['city','region','city_tier'])
    .agg(
        total_revenue      = ('revenue','sum'),
        total_orders       = ('sale_id','count'),
        total_units        = ('quantity','sum'),
        avg_margin         = ('gross_margin_pct','mean'),
        avg_discount       = ('discount_pct','mean'),
        return_rate        = ('is_returned','mean'),
        avg_trend_sens     = ('trend_sensitivity','mean'),
        unique_skus        = ('sku_id','nunique'),
        unique_channels    = ('channel','nunique'),
    )
    .round(2)
    .reset_index()
)

geo_perf['revenue_per_order'] = (
    geo_perf['total_revenue'] /
    geo_perf['total_orders']
).round(2)

geo_perf['return_rate_pct'] = (
    geo_perf['return_rate'] * 100
).round(2)

geo_perf['city_label'] = (
    geo_perf['city'] + ' (T' +
    geo_perf['city_tier'].astype(str) + ')'
)

# ── Season Performance ─────────────────────────────────────────
season_perf = (df_sales.groupby(['season','category','brand'])
    .agg(
        total_revenue  = ('revenue','sum'),
        total_orders   = ('sale_id','count'),
        avg_margin     = ('gross_margin_pct','mean'),
        avg_discount   = ('discount_pct','mean'),
        return_rate    = ('is_returned','mean'),
        total_units    = ('quantity','sum'),
    )
    .round(2)
    .reset_index()
)

season_perf['return_rate_pct'] = (
    season_perf['return_rate'] * 100
).round(2)

# ── Save ───────────────────────────────────────────────────────
supplier_perf.to_csv(
    os.path.join(PROCESSED, 'pbi_12_supplier_performance.csv'), index=False)
po_timeline.to_csv(
    os.path.join(PROCESSED, 'pbi_13_po_timeline.csv'), index=False)
geo_perf.to_csv(
    os.path.join(PROCESSED, 'pbi_14_geo_performance.csv'), index=False)
season_perf.to_csv(
    os.path.join(PROCESSED, 'pbi_15_season_performance.csv'), index=False)

print(f"Supplier Performance exported  ✅  {len(supplier_perf):,} rows")
print(f"PO Timeline exported           ✅  {len(po_timeline):,} rows")
print(f"Geographic Performance exported✅  {len(geo_perf):,} rows")
print(f"Season Performance exported    ✅  {len(season_perf):,} rows")

print(f"\n  Supplier Grade Distribution:")
print(supplier_perf.groupby('supplier_grade')['total_pos']
      .sum().to_string())

print(f"\n  Top 5 Cities by Revenue:")
top5 = geo_perf.sort_values('total_revenue', ascending=False).head(5)
for _, row in top5.iterrows():
    print(f"    {row['city']:<15} ₹{row['total_revenue']/1e6:.1f}M  "
          f"(Tier {row['city_tier']})")

Supplier Performance exported  ✅  48 rows
PO Timeline exported           ✅  1,055 rows
Geographic Performance exported✅  18 rows
Season Performance exported    ✅  240 rows

  Supplier Grade Distribution:
supplier_grade
B — Average    7136
C — Risky       864

  Top 5 Cities by Revenue:
    Kolkata         ₹33.2M  (Tier 1)
    Bangalore       ₹33.1M  (Tier 1)
    Delhi           ₹32.9M  (Tier 1)
    Chennai         ₹32.5M  (Tier 1)
    Mumbai          ₹32.2M  (Tier 1)


In [9]:
# ── Verify all Power BI files exist ───────────────────────────
pbi_files = [
    'pbi_01_kpi_summary.csv',
    'pbi_02_weekly_revenue.csv',
    'pbi_03_brand_performance.csv',
    'pbi_04_channel_performance.csv',
    'pbi_05_trend_phase_summary.csv',
    'pbi_06_trend_weekly.csv',
    'pbi_07_social_summary.csv',
    'pbi_08_inventory_health.csv',
    'pbi_09_inventory_weekly.csv',
    'pbi_10_gap_analysis.csv',
    'pbi_11_ml_scores.csv',
    'pbi_12_supplier_performance.csv',
    'pbi_13_po_timeline.csv',
    'pbi_14_geo_performance.csv',
    'pbi_15_season_performance.csv',
]

print("=" * 60)
print("  POWER BI DATA EXPORT — COMPLETE")
print("=" * 60)

total_rows = 0
total_size = 0

for fname in pbi_files:
    fpath = os.path.join(PROCESSED, fname)
    if os.path.exists(fpath):
        df_temp  = pd.read_csv(fpath)
        fsize_kb = os.path.getsize(fpath) / 1024
        total_rows += len(df_temp)
        total_size += fsize_kb
        print(f"  ✅ {fname:<40} {len(df_temp):>6,} rows  {fsize_kb:>8.1f} KB")
    else:
        print(f"  ❌ {fname:<40} NOT FOUND")

print("-" * 60)
print(f"  {'TOTAL':<40} {total_rows:>6,} rows  {total_size:>8.1f} KB")
print("=" * 60)

print(f"""
  Dashboard Pages to Build in Power BI:
  ───────────────────────────────────────
  Page 1 → Executive Summary         (pbi_01, pbi_02)
  Page 2 → Trend Intelligence        (pbi_05, pbi_06, pbi_07)
  Page 3 → Inventory Gap Analysis    (pbi_08, pbi_09, pbi_10)
  Page 4 → ML Risk Scores            (pbi_11)
  Page 5 → Brand & Channel           (pbi_03, pbi_04)
  Page 6 → Geographic Intelligence   (pbi_14)
  Page 7 → Supplier & Supply Chain   (pbi_12, pbi_13)
  Page 8 → Seasonal Intelligence     (pbi_15)
  ───────────────────────────────────────
  All files saved to: Data/Processed/
""")

print("=" * 60)
print("  Ready for Power BI ✅")
print("=" * 60)

  POWER BI DATA EXPORT — COMPLETE
  ✅ pbi_01_kpi_summary.csv                        1 rows       0.5 KB
  ✅ pbi_02_weekly_revenue.csv                   106 rows      13.2 KB
  ✅ pbi_03_brand_performance.csv                 60 rows       6.1 KB
  ✅ pbi_04_channel_performance.csv               36 rows       3.1 KB
  ✅ pbi_05_trend_phase_summary.csv               93 rows       6.8 KB
  ✅ pbi_06_trend_weekly.csv                   2,080 rows     173.1 KB
  ✅ pbi_07_social_summary.csv                   360 rows      20.8 KB
  ✅ pbi_08_inventory_health.csv                 240 rows      14.4 KB
  ✅ pbi_09_inventory_weekly.csv                 630 rows      34.8 KB
  ✅ pbi_10_gap_analysis.csv                     757 rows      95.3 KB
  ✅ pbi_11_ml_scores.csv                      1,170 rows     257.4 KB
  ✅ pbi_12_supplier_performance.csv              48 rows       4.7 KB
  ✅ pbi_13_po_timeline.csv                    1,055 rows      63.1 KB
  ✅ pbi_14_geo_performance.csv                   18 rows

In [4]:
# ── Reload products and merge segment ─────────────────────────
df_products = pd.read_csv(os.path.join(RAW, 'products.csv'))

# Merge segment into sales
df_sales = df_sales.merge(
    df_products[['sku_id','segment']],
    on='sku_id',
    how='left'
)

print(f"Segment merged ✅")
print(f"Columns now: {[c for c in df_sales.columns if c in ['brand','category','segment','year']]}")

Segment merged ✅
Columns now: ['brand', 'category', 'year', 'segment']


In [5]:
# ── Add Year column to brand_performance ──────────────────────
df_sales['sale_date'] = pd.to_datetime(df_sales['sale_date'])

brand_perf_yearly = (df_sales.groupby(['brand','category','year'])
    .agg(
        total_revenue     = ('revenue','sum'),
        total_profit      = ('gross_profit','sum'),
        total_orders      = ('sale_id','count'),
        total_units       = ('quantity','sum'),
        avg_margin        = ('gross_margin_pct','mean'),
        avg_discount      = ('discount_pct','mean'),
        avg_mrp           = ('mrp','mean'),
        avg_selling_price = ('selling_price','mean'),
        total_returns     = ('is_returned','sum'),
        return_rate       = ('is_returned','mean'),
    )
    .round(2)
    .reset_index()
)

brand_perf_yearly['revenue_share_pct'] = (
    brand_perf_yearly['total_revenue'] /
    brand_perf_yearly['total_revenue'].sum() * 100
).round(2)

brand_perf_yearly['profit_margin_pct'] = (
    brand_perf_yearly['total_profit'] /
    brand_perf_yearly['total_revenue'] * 100
).round(2)

brand_perf_yearly['return_rate_pct'] = (
    brand_perf_yearly['return_rate'] * 100
).round(2)

brand_perf_yearly.to_csv(
    os.path.join(PROCESSED, 'pbi_03_brand_performance.csv'), index=False)

print(f"Brand Performance with Year exported ✅  {len(brand_perf_yearly):,} rows")
print(brand_perf_yearly['year'].value_counts().to_string())

Brand Performance with Year exported ✅  120 rows
year
2023    60
2024    60


In [2]:
import pandas as pd
import numpy as np
import os

RAW       = r"D:\Projects\End-to-end projects\11. Fashion Trend Inventory\Data\Raw"
PROCESSED = r"D:\Projects\End-to-end projects\11. Fashion Trend Inventory\Data\Processed"

df_sales = pd.read_csv(os.path.join(RAW, 'sales_transactions.csv'))
df_sales['sale_date'] = pd.to_datetime(df_sales['sale_date'])
df_sales['year']      = df_sales['sale_date'].dt.year

print(f"df_sales reloaded ✅  {len(df_sales):,} rows")
print(f"Columns: {list(df_sales.columns)}")

df_sales reloaded ✅  120,000 rows
Columns: ['sale_id', 'sku_id', 'brand', 'category', 'subcategory', 'sale_date', 'month', 'year', 'season', 'city', 'region', 'city_tier', 'channel', 'quantity', 'mrp', 'discount_pct', 'selling_price', 'revenue', 'cogs', 'gross_profit', 'gross_margin_pct', 'is_returned', 'trend_sensitivity']


In [6]:
# ── Rebuild brand_performance with segment & year ─────────────
brand_perf_final = (df_sales.groupby(['brand','category','segment','year'])
    .agg(
        total_revenue     = ('revenue','sum'),
        total_profit      = ('gross_profit','sum'),
        total_orders      = ('sale_id','count'),
        total_units       = ('quantity','sum'),
        avg_margin        = ('gross_margin_pct','mean'),
        avg_discount      = ('discount_pct','mean'),
        avg_mrp           = ('mrp','mean'),
        avg_selling_price = ('selling_price','mean'),
        total_returns     = ('is_returned','sum'),
        return_rate       = ('is_returned','mean'),
    )
    .round(2)
    .reset_index()
)

brand_perf_final['revenue_share_pct'] = (
    brand_perf_final['total_revenue'] /
    brand_perf_final['total_revenue'].sum() * 100
).round(2)

brand_perf_final['profit_margin_pct'] = (
    brand_perf_final['total_profit'] /
    brand_perf_final['total_revenue'] * 100
).round(2)

brand_perf_final['return_rate_pct'] = (
    brand_perf_final['return_rate'] * 100
).round(2)

brand_perf_final.to_csv(
    os.path.join(PROCESSED, 'pbi_03_brand_performance.csv'), index=False)

print(f"Brand Performance rebuilt ✅  {len(brand_perf_final):,} rows")
print(f"Columns: {list(brand_perf_final.columns)}")

Brand Performance rebuilt ✅  120 rows
Columns: ['brand', 'category', 'segment', 'year', 'total_revenue', 'total_profit', 'total_orders', 'total_units', 'avg_margin', 'avg_discount', 'avg_mrp', 'avg_selling_price', 'total_returns', 'return_rate', 'revenue_share_pct', 'profit_margin_pct', 'return_rate_pct']
